# 3.02 多模态与动作分块

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/03_imitation/3.02_multimodality.ipynb)


## 为什么要学

人类示教天然是**多模态**的：同一个场景，有时从左边抓、有时从右边抓；线头有时先对齐再插、有时边插边调整。如果模型只能表示"一个答案"，它就会在多个正确答案之间**取平均**，而平均往往是错的。

另一方面，现代 VLA（ACT、Diffusion Policy、π0）都一次输出**一段动作（action chunk）**而不是一步。本课揭示一个关键事实：**动作分块让多模态问题变得更严重**，这就是为什么它们都必须配合"能表示多峰分布"的生成模型（CVAE、扩散、流匹配）。

---

## 1. 双峰示教

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from vla_ladder.envs import ObstacleReachEnv, collect_demos
from vla_ladder.utils import set_seed

env = ObstacleReachEnv(num_envs=100, seed=0)
demos = collect_demos(env, n_episodes=200, seed=0)
env.plot([d["obs"] for d in demos[:40]], title="expert demos: 50% left, 50% right")
plt.show()
print("示教成功率:", np.mean([d["success"] for d in demos]), " 总帧数:", sum(len(d["obs"]) for d in demos))

起点完全相同，专家一半从左绕、一半从右绕。在起点处，"正确动作"的分布是**双峰**的：往左上或往右上。

## 2. 单步 MSE：靠"反馈"侥幸成功

In [ ]:
def mlp(i, o, h=256):
    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, o))

def train(model, loss_fn, X, Y, steps=4000, lr=1e-3, seed=0):
    set_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X, Y = torch.as_tensor(X), torch.as_tensor(Y)
    for _ in range(steps):
        i = torch.randint(0, len(X), (256,))
        loss = loss_fn(model, X[i], Y[i])
        opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def evaluate(act_fn, H=1, exec_steps=1, n=200, seed=7):
    """act_fn(obs (n,2)) -> 动作块 (n, H*2)；每次执行前 exec_steps 步再重新预测"""
    env = ObstacleReachEnv(num_envs=n, seed=seed)
    obs = env.reset(); trajs = [obs.copy()]
    while not env.done.all():
        chunk = act_fn(torch.as_tensor(obs)).numpy().reshape(n, H, 2)
        for k in range(exec_steps):
            obs, _, _, info = env.step(chunk[:, k]); trajs.append(obs.copy())
    return info["success"].mean(), info["collided"].mean(), np.stack(trajs, 1)

O = np.concatenate([d["obs"] for d in demos]); A = np.concatenate([d["actions"] for d in demos])
mse_loss = lambda m, x, y: ((m(x) - y) ** 2).mean()
pi_step = train(mlp(2, 2), mse_loss, O, A)
sr, col, trajs = evaluate(pi_step)
print(f"单步 MSE 策略：成功率 {sr:.2f}，碰撞率 {col:.2f}")
with torch.no_grad():
    print("起点处预测的动作:", pi_step(torch.tensor([[0.0, -0.8]])).numpy().round(2), "（≈ 两个模式的平均：几乎正对障碍物）")
env.plot(list(trajs[:40]), title=f"single-step MSE BC  success={sr:.2f}"); plt.show()

单步 MSE 居然大多成功了！原因是**闭环反馈打破了对称**：起点处它确实预测了"平均动作"（直冲障碍物），但网络输出不可能完全对称，第一步稍微偏左一点，下一个状态就落在"左边的数据"里，策略就会继续往左。**每一步都重新看状态**，给了它纠正的机会。

## 3. 动作分块：为什么要一次预测一段动作？

ACT 论文（Zhao et al., 2023）提出，π0、Diffusion Policy 都沿用了动作分块。每次预测未来 $H$ 步 $\mathbf{a}_{t:t+H}$，执行其中一部分再重新预测。好处：

1. **减少复合误差**：决策次数减少为 $1/H$，有效"任务长度"变短（上一课的 $O(T^2\epsilon)$）；
2. **动作平滑、时间一致**：一段动作一起生成，不会每步抖动；
3. **对推理延迟友好**：VLA 推理一次要几十到几百毫秒，一次给出 50 步（1 秒）动作，机器人可以连续执行；
4. **处理人类示教中的停顿等非马尔可夫行为**。

但代价是：**在一个块内是"开环"执行的**，失去了上面那种每步反馈纠错的机会。

### 练习 1：构造动作块数据集

（与 0.2 课的 `chunk_actions` 相同，这里对每条轨迹做一次，再拼起来。）

In [ ]:
def make_chunk_dataset(demos, H):
    """返回 obs (N, 2) 与 chunks (N, H*2)：每帧对应从它开始的 H 步动作，越界用最后一个动作补齐"""
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def make_chunk_dataset(demos, H):
    """返回 obs (N, 2) 与 chunks (N, H*2)：每帧对应从它开始的 H 步动作，越界用最后一个动作补齐"""
    Os, Cs = [], []
    for d in demos:
        T = len(d["actions"])
        idx = np.minimum(np.arange(T)[:, None] + np.arange(H)[None, :], T - 1)   # (T, H)
        Os.append(d["obs"]); Cs.append(d["actions"][idx].reshape(T, -1))
    return np.concatenate(Os), np.concatenate(Cs)

In [ ]:
# ✅ 自测
H = 8
Oc, Ac = make_chunk_dataset(demos, H)
assert Oc.shape[0] == Ac.shape[0] == len(O) and Ac.shape[1] == 2 * H
assert np.allclose(Ac[0, :2], demos[0]["actions"][0]) and np.allclose(Ac[0, 2:4], demos[0]["actions"][1])
print("通过 ✅")

## 4. MSE + 动作分块 = 灾难

In [ ]:
pi_chunk = train(mlp(2, 2 * H), mse_loss, Oc, Ac)
sr, col, trajs_c = evaluate(pi_chunk, H=H, exec_steps=H // 2)
print(f"MSE 动作块策略：成功率 {sr:.2f}，碰撞率 {col:.2f}")
with torch.no_grad():
    c0 = pi_chunk(torch.tensor([[0.0, -0.8]])).numpy().reshape(H, 2)
print("起点处预测的动作块（每一步都是平均动作）:\n", c0.round(2))
env.plot(list(trajs_c[:40]), title=f"MSE + action chunking  success={sr:.2f}"); plt.show()

起点处 MSE 预测了一整段"平均动作"，开环执行 4 步后已经正对障碍物冲了过去，之后再怎么预测都来不及了。**模型必须能表示"要么左、要么右"，而不是"左右平均"**。

---

## 5. 混合密度网络（MDN）：能表示多峰的最简单模型

让网络输出 $K$ 个高斯分量的**混合**：

$$
p(\mathbf{a}|o)=\sum_{k=1}^K w_k(o)\,\mathcal{N}\big(\mathbf{a};\boldsymbol\mu_k(o),\text{diag}(\boldsymbol\sigma_k^2(o))\big),\qquad \sum_k w_k=1
$$

训练最小化负对数似然。用 log-sum-exp 保证数值稳定：

$$
-\log p(\mathbf{a}|o)=-\log\sum_k\exp\Big(\log w_k+\log\mathcal{N}(\mathbf{a};\boldsymbol\mu_k,\boldsymbol\sigma_k)\Big)
$$

采样：先按 $w_k$ 选一个分量 $k$，再取它的均值（或从中采样）。**选分量这一步就是"决定从左还是从右"**。

### 练习 2：MDN 的负对数似然与采样

In [ ]:
class MDN(nn.Module):
    def __init__(self, obs_dim, act_dim, K=4, hidden=256):
        super().__init__()
        self.K, self.act_dim = K, act_dim
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.logits = nn.Linear(hidden, K)                  # 混合权重 w_k 的 logits
        self.mu = nn.Linear(hidden, K * act_dim)
        self.log_std = nn.Linear(hidden, K * act_dim)

    def forward(self, obs):
        h = self.body(obs)
        B = obs.shape[0]
        return (self.logits(h), self.mu(h).view(B, self.K, self.act_dim),
                self.log_std(h).view(B, self.K, self.act_dim).clamp(-7, 2))

    def nll(self, obs, act):
        """act: (B, act_dim) -> 标量"""
        logits, mu, log_std = self(obs)
        # TODO:
        # 1) log_w = log_softmax(logits)                                    (B, K)
        # 2) log_n = Normal(mu, exp(log_std)).log_prob(act[:, None]).sum(-1)  (B, K)：每个分量的对数密度
        # 3) return -logsumexp(log_w + log_n, dim=-1).mean()
        raise NotImplementedError

    @torch.no_grad()
    def sample(self, obs):
        """按权重选分量，返回该分量的均值 (B, act_dim)"""
        logits, mu, _ = self(obs)
        # TODO: k = Categorical(logits=logits).sample()；返回 mu[arange(B), k]
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class MDN(nn.Module):
    def __init__(self, obs_dim, act_dim, K=4, hidden=256):
        super().__init__()
        self.K, self.act_dim = K, act_dim
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.logits = nn.Linear(hidden, K)
        self.mu = nn.Linear(hidden, K * act_dim)
        self.log_std = nn.Linear(hidden, K * act_dim)

    def forward(self, obs):
        h = self.body(obs)
        B = obs.shape[0]
        return (self.logits(h), self.mu(h).view(B, self.K, self.act_dim),
                self.log_std(h).view(B, self.K, self.act_dim).clamp(-7, 2))

    def nll(self, obs, act):
        """act: (B, act_dim) -> 标量"""
        logits, mu, log_std = self(obs)
        log_w = torch.log_softmax(logits, dim=-1)
        log_n = torch.distributions.Normal(mu, log_std.exp()).log_prob(act[:, None]).sum(-1)
        return -torch.logsumexp(log_w + log_n, dim=-1).mean()

    @torch.no_grad()
    def sample(self, obs):
        """按权重选分量，返回该分量的均值 (B, act_dim)"""
        logits, mu, _ = self(obs)
        k = torch.distributions.Categorical(logits=logits).sample()
        return mu[torch.arange(obs.shape[0]), k]

In [ ]:
# ✅ 自测：一维双峰数据，MDN 应该学出两个分量
set_seed(0)
x = torch.zeros(2000, 1); y = torch.where(torch.rand(2000, 1) < 0.5, -1.0, 1.0) + 0.05 * torch.randn(2000, 1)
m = MDN(1, 1, K=2)
opt = torch.optim.Adam(m.parameters(), lr=1e-2)
for _ in range(500):
    loss = m.nll(x, y); opt.zero_grad(); loss.backward(); opt.step()
lg, mu, _ = m(x[:1])
print("两个分量的权重:", torch.softmax(lg, -1).detach().numpy().round(2), " 均值:", mu.detach().numpy().ravel().round(2))
assert sorted(mu.detach().numpy().ravel().round(1).tolist()) == [-1.0, 1.0]
print("通过 ✅（MSE 在这里只会学出 0）")

### 用 MDN 学动作块

In [ ]:
mdn = train(MDN(2, 2 * H, K=4), lambda m, x, y: m.nll(x, y), Oc, Ac)
sr_m, col_m, trajs_m = evaluate(mdn.sample, H=H, exec_steps=H // 2)
print(f"MDN 动作块策略：成功率 {sr_m:.2f}，碰撞率 {col_m:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
env.plot(list(trajs_c[:40]), title=f"MSE chunk  success={sr:.2f}", ax=ax[0])
env.plot(list(trajs_m[:40]), title=f"MDN chunk  success={sr_m:.2f}", ax=ax[1])
plt.show()
assert sr_m > sr + 0.3

MDN 每次在起点"抛硬币"决定方向，然后**整段动作都朝那个方向**，于是能绕开障碍物。

> **仍然存在的问题：模式切换。** 如果在两个模式都合理的区域里连续采样几个动作块，可能这次选左、下次选右，轨迹会左右摇摆（试试把 `exec_steps` 改成 1 看看）。ACT 用**时间集成**（temporal ensembling）、Diffusion Policy 用**执行多步再重规划 + 观测历史**、π0 用**大块（50 步）+ 实时分块（RTC）**来缓解它。

---

## 小结

| 方法 | 能表示多峰？ | 动作块下的表现 |
|---|---|---|
| MSE（高斯，单峰） | ❌ 取平均 | 灾难（开环执行"平均动作"） |
| 离散化 + 交叉熵（RT-2） | ✅ 每维独立的多峰 | 维度之间的相关性难以表示 |
| 混合密度网络 | ✅ 但分量数 K 要人为选择 | 可以，但高维时训练不稳定 |
| CVAE（ACT，第 5 章） | ✅ 用隐变量 $z$ 选模式 | 好 |
| 扩散 / 流匹配（第 6、7 章） | ✅ 任意复杂分布 | 好，**π0 的选择** |

**核心结论**：动作分块 + 能表示多峰分布的生成模型 = 现代 VLA 的动作头。接下来两章依次实现 CVAE（ACT）与扩散/流匹配。在那之前，我们需要先掌握 Transformer（第 4 章）。